In [12]:
import pandas as pd
import os
from langchain_community.document_loaders import DataFrameLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# APIキーの設定
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [13]:
# データの読み込み
df = pd.read_csv("./cosmetics.csv")
df.head()

,Label,Brand,Name,Price,Rank,Ingredients,Combination,Dry,Normal,Oily,Sensitive
0,Moisturizer,LA MER,Crème de la Mer,175,4.1,"Algae (Seaweed) Extract, Mineral Oil, Petrolat...",1,1,1,1,1
1,Moisturizer,SK-II,Facial Treatment Essence,179,4.1,"Galactomyces Ferment Filtrate (Pitera), Butyle...",1,1,1,1,1
2,Moisturizer,DRUNK ELEPHANT,Protini™ Polypeptide Cream,68,4.4,"Water, Dicaprylyl Carbonate, Glycerin, Ceteary...",1,1,1,1,0
3,Moisturizer,LA MER,The Moisturizing Soft Cream,175,3.8,"Algae (Seaweed) Extract, Cyclopentasiloxane, P...",1,1,1,1,1
4,Moisturizer,IT COSMETICS,Your Skin But Better™ CC+™ Cream with SPF 50+,38,4.1,"Water, Snail Secretion Filtrate, Phenyl Trimet...",1,1,1,1,1


In [16]:
df = df.dropna(subset=["Name", "Ingredients"])

In [17]:
# オリジナル列を作成
df['ai_context'] = (
    "ブランド: " + df['Brand'] + "\n" +
    "商品名: " + df['Name'] + " (" + df['Label'] + ")\n" +
    "成分: " + df['Ingredients']
)

In [18]:
# LangChain用に変換
loader = DataFrameLoader(df, page_content_column="ai_context")
documents = loader.load()

In [19]:
# データベース(FAISS)の構築
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [20]:
# LLMとプロンプトの設定
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
prompt = ChatPromptTemplate.from_template("""

あなたは美容・コスメ専門のプロフェッショナルなAIエンジニア兼アドバイザーです。
以下の参考情報（Sephoraのコスメ成分データベース）のみを用いて、ユーザーの質問に答えてください。
提案する際は、ブランド名と商品名、そしてなぜその成分が良いのかを論理的に説明してください。

<参考情報>
{context}
</参考情報>

ユーザーの質問: {input}
""")

In [21]:
# 検索してきたドキュメントの文章だけを綺麗に結合する関数
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [22]:
# ログチェインの構築（パイプラインで繋ぐ）
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [26]:
# 質問の設定
query = "乾燥肌(Dry)で悩んでいるのですが、保湿成分（Glycerinなど）が含まれたおすすめの化粧水(Moisturizer)はありますか？"

In [30]:
query2 = "ニキビができやすい肌は、なんの成分が必要？"

In [31]:
# 実行
print(f"質問:\n {query2}\n")
try:
    response = rag_chain.invoke(query2)
    print("AIの回答:\n", response)
except Exception as e:
    print("\nエラーが発生しました。APIキーが正しく設定されているか確認してください。")
    print("詳細:", e)

質問:
 ニキビができやすい肌は、なんの成分が必要？

AIの回答:
 ニキビができやすい肌には、殺菌作用や炎症を抑える作用がある成分が必要です。そのような成分として、以下の成分が効果的です。

1. Lauric Acid: ニキビを引き起こすアクネ菌を殺菌する効果があります。
2. Trehalose: 皮脂の過剰分泌を抑え、肌のバリア機能を強化する効果があります。
3. Betaine: 保湿効果があり、肌の水分バランスを整えることでニキビの予防に役立ちます。

以上の成分が含まれている商品として、SHISEIDOのIbuki Gentle Cleanserがおすすめです。このクレンザーは、ニキビができやすい肌に必要な成分を含んでおり、肌を優しく洗浄しながらニキビの予防に効果的です。
